In [ ]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import torchvision

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.11.0+cu128
transformers: 5.10.1
torchvision: 0.26.0+cu128


In [ ]:
import os
import re
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# Stage 1: Pseudo-Label Generation

The Support Integrity Auditor must operate without manually annotated mismatch labels. To bootstrap supervision, we first estimate the intrinsic severity of each ticket independent of its human-assigned priority.

The first signal uses a Large Language Model (LLM) in a zero-shot setting. The model evaluates ticket subject, description, issue category and channel, then assigns a severity score on a 4-level scale:

- 0 → Low
- 1 → Medium
- 2 → High
- 3 → Critical

This severity estimate acts as a semantic understanding signal and forms the foundation of the pseudo-labeling pipeline.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

USE_PRECOMPUTED_LLM = True

RAW_DATA_PATH = "/content/customer_support_tickets.csv"
LLM_SCORE_PATH = "/content/llm_scores.csv"

CHECKPOINT_DIR = "/content/deberta_checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

SEVERITY_MAP = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

INV_SEVERITY_MAP = {
    0: "Low",
    1: "Medium",
    2: "High",
    3: "Critical"
}

In [ ]:
df = pd.read_csv(
    RAW_DATA_PATH
)

print("Shape:", df.shape)

df.head()

Shape: (20000, 12)


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["Ticket_Subject"] = (
    df["Ticket_Subject"]
    .fillna("")
    .astype(str)
)

df["Ticket_Description"] = (
    df["Ticket_Description"]
    .fillna("")
    .astype(str)
)

df["full_text"] = (

    df["Ticket_Subject"]
    + " "

    + df["Ticket_Description"]

).apply(clean_text)

df[
    [
        "Ticket_Subject",
        "Ticket_Description",
        "full_text"
    ]
].head()

,Ticket_Subject,Ticket_Description,full_text
0,Hours of operation - Individual,"Hi Support, Where is your headquarters located...","Hours of operation - Individual Hi Support, Wh..."
1,Data not syncing - Card,"Hi Support, The application crashes every time...","Data not syncing - Card Hi Support, The applic..."
2,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...","2FA issues - Question Hi Support, How do I upg..."
3,Login failed - Let,"Hi Support, The dashboard is not loading any d...","Login failed - Let Hi Support, The dashboard i..."
4,Refund status - Attention,"Hi Support, I have been trying to update my pa...","Refund status - Attention Hi Support, I have b..."


In [ ]:
import os
from tqdm import tqdm

if USE_PRECOMPUTED_LLM:

    print(
        "Loading cached LLM scores..."
    )

    try:
        llm_df = pd.read_csv(
            LLM_SCORE_PATH
        )

        print(
            llm_df.shape
        )

    except FileNotFoundError:
        print("llm_scores.csv not found. Computing LLM scores...")

        from transformers import (
            AutoTokenizer,
            AutoModelForCausalLM
        )

        MODEL_NAME = (
            "microsoft/Phi-3-mini-4k-instruct"
        )

        tokenizer = (
            AutoTokenizer
            .from_pretrained(
                MODEL_NAME
            )
        )

        model = (
            AutoModelForCausalLM
            .from_pretrained(
                MODEL_NAME
            )
        )

        def build_prompt(
            subject,
            description,
            category,
            channel
        ):

            return f"""
You are a support ticket severity auditor.

Assign a severity score.

Severity scale:

0 = Low
1 = Medium
2 = High
3: Critical

Guidelines:

3:
Security incidents,
fraud,
service outages,
major operational disruption.

2:
Major functionality failures,
API failures,
application crashes,
login failures,
payment issues.

1:
User-impacting issues with workarounds.

0:
Questions,
clarifications,
information requests,
minor requests.

Ticket Category:
{category}

Ticket Channel:
{channel}

Subject:
{subject}

Description:
{description}

Return ONLY ONE NUMBER.

Severity Score:
"""

        def predict_severity(
            subject,
            description,
            category,
            channel
        ):

            prompt = build_prompt(
                subject,
                description,
                category,
                channel
            )

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id
            )

            new_tokens = outputs[0][
                inputs["input_ids"].shape[1]:
            ]

            response = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True
            ).strip()

            numbers = re.findall(
                r"\b([0-3])\b",
                response
            )

            if len(numbers) > 0:
                return int(numbers[0])

            return None

        CHECKPOINT_FILE = "llm_scores_checkpoint.csv"
        SAVE_EVERY = 100

        if os.path.exists(CHECKPOINT_FILE):
            checkpoint = pd.read_csv(CHECKPOINT_FILE)
            completed_ids = set(checkpoint["Ticket_ID"])
            remaining_df = df[
                ~df["Ticket_ID"].isin(completed_ids)
            ].copy()
            results = checkpoint.to_dict("records")
            print(f"Resuming...")
            print(f"Completed: {len(checkpoint)}")
            print(f"Remaining: {len(remaining_df)}")
        else:
            remaining_df = df.copy()
            results = []
            print("Starting fresh...")

        counter = 0
        for _, row in tqdm(
            remaining_df.iterrows(),
            total=len(remaining_df)
        ):
            try:
                score = predict_severity(
                    row["Ticket_Subject"],
                    row["Ticket_Description"],
                    row["Issue_Category"],
                    row["Ticket_Channel"]
                )
            except Exception as e:
                print("ERROR:", e)
                score = None

            results.append({
                "Ticket_ID": row["Ticket_ID"],
                "llm_score": score
            })

            counter += 1
            if counter % SAVE_EVERY == 0:
                pd.DataFrame(results).to_csv(
                    CHECKPOINT_FILE,
                    index=False
                )

        llm_df = pd.DataFrame(results)
        llm_df.to_csv(
            "llm_scores.csv",
            index=False
        )
        print("\nDone!")
        print(llm_df.shape)

else:

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM
    )

    MODEL_NAME = (
        "microsoft/Phi-3-mini-4k-instruct"
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            MODEL_NAME
        )
    )

    model = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_NAME
        )
    )

    def build_prompt(
        subject,
        description,
        category,
        channel
    ):

        return f"""
You are a support ticket severity auditor.

Assign a severity score.

Severity scale:

0 = Low
1 = Medium
2 = High
3: Critical

Guidelines:

3:
Security incidents,
fraud,
service outages,
major operational disruption.

2:
Major functionality failures,
API failures,
application crashes,
login failures,
payment issues.

1:
User-impacting issues with workarounds.

0:
Questions,
clarifications,
information requests,
minor requests.

Ticket Category:
{category}

Ticket Channel:
{channel}

Subject:
{subject}

Description:
{description}

Return ONLY ONE NUMBER.

Severity Score:
"""

    def predict_severity(
        subject,
        description,
        category,
        channel
    ):

        prompt = build_prompt(
            subject,
            description,
            category,
            channel
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

        new_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        numbers = re.findall(
            r"\b([0-3])\b",
            response
        )

        if len(numbers) > 0:
            return int(numbers[0])

        return None

    CHECKPOINT_FILE = "llm_scores_checkpoint.csv"
    SAVE_EVERY = 100

    if os.path.exists(CHECKPOINT_FILE):
        checkpoint = pd.read_csv(CHECKPOINT_FILE)
        completed_ids = set(checkpoint["Ticket_ID"])
        remaining_df = df[
            ~df["Ticket_ID"].isin(completed_ids)
        ].copy()
        results = checkpoint.to_dict("records")
        print(f"Resuming...")
        print(f"Completed: {len(checkpoint)}")
        print(f"Remaining: {len(remaining_df)}")
    else:
        remaining_df = df.copy()
        results = []
        print("Starting fresh...")

    counter = 0
    for _, row in tqdm(
        remaining_df.iterrows(),
        total=len(remaining_df)
    ):
        try:
            score = predict_severity(
                row["Ticket_Subject"],
                row["Ticket_Description"],
                row["Issue_Category"],
                row["Ticket_Channel"]
            )
        except Exception as e:
            print("ERROR:", e)
            score = None

        results.append({
            "Ticket_ID": row["Ticket_ID"],
            "llm_score": score
        })

        counter += 1
        if counter % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(
                CHECKPOINT_FILE,
                index=False
            )

    llm_df = pd.DataFrame(results)
    llm_df.to_csv(
        "llm_scores.csv",
        index=False
    )
    print("\nDone!")
    print(llm_df.shape)

df = df.merge(
    llm_df,
    on="Ticket_ID",
    how="left"
)

df[
    [
        "Ticket_ID",
        "llm_score"
    ]
].head()

Loading cached LLM scores...
(20000, 2)


,Ticket_ID,llm_score
0,TKT-100000,0
1,TKT-100001,2
2,TKT-100002,0
3,TKT-100003,2
4,TKT-100004,1


In [ ]:
print(
    llm_df["llm_score"]
    .value_counts()
)

llm_score
0    12464
2     4870
1     1565
3     1101
Name: count, dtype: int64


Note: Initially the pseudo_labels.csv was generated locally using generate_pesudo_labels.py,the following cells follow the same methodology as used in src/signal/ folder files for regenration but it is recommended to use the original one in order to avoid difference in train,test,eval set which may lead to varying accuracy as the model was trained and evaluated on those datasets do upload them during run time.

# Semantic Embedding Signal

While the LLM provides ticket-level reasoning, severity patterns also emerge from groups of semantically similar tickets.

To capture this structure, ticket text is converted into dense vector embeddings using Sentence Transformers (all-MiniLM-L6-v2). These embeddings encode contextual meaning beyond simple keyword matching.

The resulting vectors are later clustered to identify latent groups of tickets exhibiting similar urgency characteristics.

In [ ]:
embedding_model = (
    SentenceTransformer(
        "all-MiniLM-L6-v2"
    )
)

embeddings = (
    embedding_model.encode(
        df["full_text"].tolist(),
        show_progress_bar=True
    )
)

embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

(20000, 384)

# Embedding-Based Clustering

The generated embeddings are grouped using K-Means clustering.

The intuition is that tickets discussing similar operational issues often share similar severity levels. For each cluster, a representative severity score is computed based on the average LLM severity of tickets belonging to that cluster.

This produces an independent severity estimate that is derived from semantic neighborhood structure rather than direct LLM reasoning.

In [ ]:
N_CLUSTERS = 8

kmeans = KMeans(

    n_clusters=N_CLUSTERS,

    random_state=SEED,

    n_init=10
)

cluster_ids = (
    kmeans.fit_predict(
        embeddings
    )
)

df["cluster_id"] = (
    cluster_ids
)

cluster_severity = {}

for cluster in range(
    N_CLUSTERS
):

    cluster_rows = (
        df[
            df["cluster_id"]
            ==
            cluster
        ]
    )

    if len(cluster_rows) == 0:

        cluster_severity[
            cluster
        ] = 1

    else:

        cluster_severity[
            cluster
        ] = int(
            round(
                cluster_rows[
                    "llm_score"
                ].mean()
            )
        )

df["cluster_score"] = (
    df["cluster_id"]
    .map(
        cluster_severity
    )
)

df[
    [
        "cluster_id",
        "cluster_score"
    ]
].head()

,cluster_id,cluster_score
0,0,0
1,2,1
2,7,0
3,2,1
4,5,0


# Resolution-Time Severity Proxy

Resolution time provides an indirect but valuable severity indicator.

Tickets requiring extended investigation and remediation often correspond to more severe operational incidents. Resolution duration is therefore converted into a severity proxy score using predefined thresholds.

This structured signal complements the semantic signals generated from ticket content.

In [ ]:
def resolution_to_severity(hours):

    if pd.isna(hours):
        return 1

    if hours <= 12:
        return 0

    elif hours <= 48:
        return 1

    elif hours <= 96:
        return 2

    else:
        return 3


df["resolution_score"] = (

    df["Resolution_Time_Hours"]

    .apply(
        resolution_to_severity
    )

)

df[
    [
        "Resolution_Time_Hours",
        "resolution_score"
    ]
].head()

,Resolution_Time_Hours,resolution_score
0,43,1
1,41,1
2,7,0
3,41,1
4,40,1


# Rule-Based Severity Signal

To improve robustness and interpretability, a lightweight rule-based severity detector is introduced.

The system searches for escalation phrases, outage indicators, authentication failures, payment issues and other high-risk keywords commonly associated with critical support incidents.

Unlike learned models, this signal remains fully explainable and contributes directly to the final evidence dossier.

In [ ]:
HIGH_KEYWORDS = [

    "outage",
    "server down",
    "critical",
    "fraud",
    "unauthorized",
    "breach",
    "payment failed",
    "login failed",
    "api failure",
    "service unavailable"
]

MEDIUM_KEYWORDS = [

    "timeout",
    "slow",
    "latency",
    "error",
    "cannot access",
    "bug",
    "issue"
]


def get_rule_score(text):

    text = str(text).lower()

    evidence = []

    score = 0

    for kw in HIGH_KEYWORDS:

        if kw in text:

            score = max(
                score,
                3
            )

            evidence.append(
                kw
            )

    for kw in MEDIUM_KEYWORDS:

        if kw in text:

            score = max(
                score,
                2
            )

            evidence.append(
                kw
            )

    return score, ", ".join(evidence)


rule_results = (

    df["full_text"]

    .apply(
        get_rule_score
    )

)

df["rule_score"] = (

    rule_results

    .apply(
        lambda x: x[0]
    )
)

df["rule_evidence"] = (

    rule_results

    .apply(
        lambda x: x[1]
    )
)

df[
    [
        "rule_score",
        "rule_evidence"
    ]
].head()

,rule_score,rule_evidence
0,0,
1,0,
2,2,issue
3,3,login failed
4,0,


In [ ]:
agreement_pairs = [

    (
        "llm_score",
        "cluster_score"
    ),

    (
        "llm_score",
        "resolution_score"
    ),

    (
        "llm_score",
        "rule_score"
    ),

    (
        "cluster_score",
        "resolution_score"
    ),

    (
        "cluster_score",
        "rule_score"
    ),

    (
        "resolution_score",
        "rule_score"
    )
]

rows = []

for a, b in agreement_pairs:

    agreement = (

        df[a]
        ==
        df[b]

    ).mean()

    rows.append(

        [
            a,
            b,
            round(
                agreement,
                4
            )
        ]
    )

agreement_df = pd.DataFrame(

    rows,

    columns=[

        "Signal A",
        "Signal B",
        "Agreement"
    ]
)

agreement_df

,Signal A,Signal B,Agreement
0,llm_score,cluster_score,0.5523
1,llm_score,resolution_score,0.2353
2,llm_score,rule_score,0.6010
3,cluster_score,resolution_score,0.3044
4,cluster_score,rule_score,0.4940
5,resolution_score,rule_score,0.2418


# Signal Agreement Analysis

Before combining signals, pairwise agreement is measured.

Agreement statistics provide insight into how consistently different severity estimation methods evaluate the same ticket. High agreement indicates stronger confidence, while disagreement highlights potentially ambiguous cases.

These measurements are also used to justify the fusion strategy required by the project specification.

In [ ]:
signal_strength = []

for signal in [

    "llm_score",
    "cluster_score",
    "resolution_score",
    "rule_score"

]:

    agreement = (

        df[signal]

        ==
        df["llm_score"]

    ).mean()

    signal_strength.append(

        [
            signal,
            round(
                agreement,
                4
            )
        ]
    )

signal_strength_df = pd.DataFrame(

    signal_strength,

    columns=[

        "Signal",
        "Agreement_With_LLM"
    ]
)

signal_strength_df

,Signal,Agreement_With_LLM
0,llm_score,1.0000
1,cluster_score,0.5523
2,resolution_score,0.2353
3,rule_score,0.6010


# Multi-Signal Severity Fusion

No single signal is sufficiently reliable in isolation.

To obtain a robust estimate of ticket severity, the following weighted fusion strategy is used:

- LLM Severity Score → 30%
- Cluster Severity Score → 30%
- Resolution-Time Signal → 25%
- Rule-Based Signal → 15%

The weighted score is rounded and mapped back to a severity category.

This fused severity estimate serves as the system's inferred ground-truth severity.

In [ ]:
fusion_score = (

    0.30 * df["llm_score"]

    +

    0.30 * df["cluster_score"]

    +

    0.25 * df["resolution_score"]

    +

    0.15 * df["rule_score"]

)

df["fusion_score"] = fusion_score


df["inferred_severity_num"] = (

    fusion_score

    .round()

    .clip(
        0,
        3
    )

    .astype(int)
)

df["inferred_severity"] = (

    df["inferred_severity_num"]

    .map(
        INV_SEVERITY_MAP
    )
)

df[
    [
        "fusion_score",
        "inferred_severity"
    ]
].head()

,fusion_score,inferred_severity
0,0.25,Low
1,1.15,Medium
2,0.30,Low
3,1.60,High
4,0.55,Medium


In [ ]:
df["assigned_severity_num"] = (

    df["Priority_Level"]

    .map(
        SEVERITY_MAP
    )
)

df["severity_delta"] = (

    df["inferred_severity_num"]

    -

    df["assigned_severity_num"]
)

df["mismatch"] = (

    df["severity_delta"]

    !=

    0

).astype(int)

# Binary Mismatch Label Construction

The inferred severity is compared against the original human-assigned priority level.

If the two severity estimates differ, the ticket is labeled as a mismatch. Otherwise, it is labeled as consistent.

Additionally, mismatches are categorized into:

- Hidden Crisis: inferred severity exceeds assigned priority.
- False Alarm: assigned priority exceeds inferred severity.

The resulting labels form the self-supervised training dataset used for classifier development.

In [ ]:
def get_mismatch_type(delta):

    if delta > 0:

        return "Hidden Crisis"

    elif delta < 0:

        return "False Alarm"

    else:

        return "Consistent"


df["mismatch_type"] = (

    df["severity_delta"]

    .apply(
        get_mismatch_type
    )
)

df[
    [
        "Priority_Level",
        "inferred_severity",
        "severity_delta",
        "mismatch_type"
    ]
].head()

,Priority_Level,inferred_severity,severity_delta,mismatch_type
0,High,Low,-2,False Alarm
1,High,Medium,-1,False Alarm
2,High,Low,-2,False Alarm
3,Low,High,2,Hidden Crisis
4,Medium,Medium,0,Consistent


In [ ]:
df["text"] = (

    df["Ticket_Subject"]

    + " "

    + df["Ticket_Description"]

).fillna("")


pseudo_cols = [

    "Ticket_ID",

    "text",

    "Priority_Level",

    "llm_score",

    "cluster_score",

    "resolution_score",

    "rule_score",

    "rule_evidence",

    "inferred_severity",

    "severity_delta",

    "mismatch",

    "mismatch_type"
]

pseudo_df = df[pseudo_cols]

pseudo_df.to_csv(

    "pseudo_labels.csv",

    index=False
)

print(
    pseudo_df.shape
)

pseudo_df.head()

(20000, 12)


,Ticket_ID,text,Priority_Level,llm_score,cluster_score,resolution_score,rule_score,rule_evidence,inferred_severity,severity_delta,mismatch,mismatch_type
0,TKT-100000,"Hours of operation - Individual Hi Support, Wh...",High,0,0,1,0,,Low,-2,1,False Alarm
1,TKT-100001,"Data not syncing - Card Hi Support, The applic...",High,2,1,1,0,,Medium,-1,1,False Alarm
2,TKT-100002,"2FA issues - Question Hi Support, How do I upg...",High,0,0,0,2,issue,Low,-2,1,False Alarm
3,TKT-100003,"Login failed - Let Hi Support, The dashboard i...",Low,2,1,1,3,login failed,High,2,1,Hidden Crisis
4,TKT-100004,"Refund status - Attention Hi Support, I have b...",Medium,1,0,1,0,,Medium,0,0,Consistent


In [ ]:
df["text"] = (

    df["Ticket_Subject"]

    + " "

    + df["Ticket_Description"]

).fillna("")


pseudo_cols = [

    "Ticket_ID",

    "text",

    "Priority_Level",

    "llm_score",

    "cluster_score",

    "resolution_score",

    "rule_score",

    "rule_evidence",

    "inferred_severity",

    "severity_delta",

    "mismatch",

    "mismatch_type"
]

pseudo_df = df[pseudo_cols]

pseudo_df.to_csv(

    "pseudo_labels.csv",

    index=False
)

print(
    pseudo_df.shape
)

pseudo_df.head()

(20000, 12)


,Ticket_ID,text,Priority_Level,llm_score,cluster_score,resolution_score,rule_score,rule_evidence,inferred_severity,severity_delta,mismatch,mismatch_type
0,TKT-100000,"Hours of operation - Individual Hi Support, Wh...",High,0,0,1,0,,Low,-2,1,False Alarm
1,TKT-100001,"Data not syncing - Card Hi Support, The applic...",High,2,1,1,0,,Medium,-1,1,False Alarm
2,TKT-100002,"2FA issues - Question Hi Support, How do I upg...",High,0,0,0,2,issue,Low,-2,1,False Alarm
3,TKT-100003,"Login failed - Let Hi Support, The dashboard i...",Low,2,1,1,3,login failed,High,2,1,Hidden Crisis
4,TKT-100004,"Refund status - Attention Hi Support, I have b...",Medium,1,0,1,0,,Medium,0,0,Consistent


# Dataset Preparation

After pseudo-label generation, the dataset is transformed into a supervised learning problem.

Each ticket is converted into a structured textual representation combining:

- Ticket metadata
- Assigned priority
- Resolution time
- Subject
- Description

The data is then split into train, validation and test partitions using stratified sampling to preserve class balance.

Note:
Kindly use the dataset in the data/processed as that was the data used to train deberta model later.




In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    pseudo_df,
    test_size=0.20,
    random_state=SEED,
    stratify=pseudo_df["mismatch"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["mismatch"]
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

train_df.to_csv(
    "train.csv",
    index=False
)

val_df.to_csv(
    "val.csv",
    index=False
)

test_df.to_csv(
    "test.csv",
    index=False
)

Train: (16000, 12)
Val: (2000, 12)
Test: (2000, 12)


# Stage 2: Fine-Tuned Mismatch Classifier

A DeBERTa-v3-small transformer model is fine-tuned to predict whether a ticket represents a priority mismatch.

The model receives both ticket text and structured metadata.

This stage allows the system to generalize beyond the pseudo-labeled examples used during supervision generation.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_PATH = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
import torch

bad = False

for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print("NaN found:", name)
        bad = True
        break

print("Model OK:", not bad)

Model OK: True


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[
        ["text", "mismatch"]
    ]
)

val_ds = Dataset.from_pandas(
    val_df[
        ["text", "mismatch"]
    ]
)

test_ds = Dataset.from_pandas(
    test_df[
        ["text", "mismatch"]
    ]
)

train_ds = train_ds.rename_column(
    "mismatch",
    "labels"
)

val_ds = val_ds.rename_column(
    "mismatch",
    "labels"
)

test_ds = test_ds.rename_column(
    "mismatch",
    "labels"
)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    pseudo_df,
    test_size=0.20,
    random_state=SEED,
    stratify=pseudo_df["mismatch"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["mismatch"]
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

train_df.to_csv(
    "train.csv",
    index=False
)

val_df.to_csv(
    "val.csv",
    index=False
)

test_df.to_csv(
    "test.csv",
    index=False
)

Train: (16000, 12)
Val: (2000, 12)
Test: (2000, 12)


In [ ]:
def tokenize(texts):

    return tokenizer(
        texts,
        truncation=True,
        max_length=256
    )

train_ds = Dataset.from_pandas(
    train_df[
        ["text", "mismatch"]
    ]
)

val_ds = Dataset.from_pandas(
    val_df[
        ["text", "mismatch"]
    ]
)

test_ds = Dataset.from_pandas(
    test_df[
        ["text", "mismatch"]
    ]
)

train_ds = train_ds.rename_column(
    "mismatch",
    "labels"
)

val_ds = val_ds.rename_column(
    "mismatch",
    "labels"
)

test_ds = test_ds.rename_column(
    "mismatch",
    "labels"
)

train_ds = train_ds.map(
    tokenize,
    batched=True,
    input_columns=["text"]
)

val_ds = val_ds.map(
    tokenize,
    batched=True,
    input_columns=["text"]
)

test_ds = test_ds.map(
    tokenize,
    batched=True,
    input_columns=["text"]
)

train_ds = train_ds.remove_columns(
    ["text"]
)

val_ds = val_ds.remove_columns(
    ["text"]
)

test_ds = test_ds.remove_columns(
    ["text"]
)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

print("Model loaded successfully")

Model loaded successfully


# Model Evaluation

Performance is measured on a held-out test set using the metrics specified in the problem statement:

- Binary Classification Accuracy
- Macro F1 Score
- Per-Class Recall

These metrics evaluate both overall predictive quality and the model's ability to detect mismatched tickets reliably.

In [ ]:
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

test_df = pd.read_csv("test.csv")

preds = []

for text in test_df["text"]:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        pred = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    preds.append(pred)

y_true = test_df["mismatch"].values

acc = accuracy_score(
    y_true,
    preds
)

macro_f1 = f1_score(
    y_true,
    preds,
    average="macro"
)

precision = precision_score(
    y_true,
    preds,
    average="macro"
)

recall = recall_score(
    y_true,
    preds,
    average="macro"
)

recall_class0 = recall_score(
    y_true,
    preds,
    pos_label=0
)

recall_class1 = recall_score(
    y_true,
    preds,
    pos_label=1
)

print("="*50)
print("FINAL TEST RESULTS")
print("="*50)

print(f"Accuracy       : {acc:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"Recall Class 0 : {recall_class0:.4f}")
print(f"Recall Class 1 : {recall_class1:.4f}")

print("\nConfusion Matrix")
print(
    confusion_matrix(
        y_true,
        preds
    )
)

print("\nClassification Report")
print(
    classification_report(
        y_true,
        preds,
        digits=4
    )
)

FINAL TEST RESULTS
Accuracy       : 0.9175
Macro F1       : 0.9161
Precision      : 0.9153
Recall         : 0.9170
Recall Class 0 : 0.9131
Recall Class 1 : 0.9208

Confusion Matrix
[[ 788   75]
 [  90 1047]]

Classification Report
              precision    recall  f1-score   support

           0     0.8975    0.9131    0.9052       863
           1     0.9332    0.9208    0.9270      1137

    accuracy                         0.9175      2000
   macro avg     0.9153    0.9170    0.9161      2000
weighted avg     0.9178    0.9175    0.9176      2000



In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import torch

MODEL_PATH = "/content/model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

print("Model Loaded")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Model Loaded


In [ ]:
import torch
import torch.nn.functional as F


def predict_mismatch(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probs = F.softmax(
            outputs.logits,
            dim=1
        )

    pred = probs.argmax().item()

    confidence = (
        probs.max().item()
    )

    return pred, confidence

# Stage 3: Evidence Dossier Generation

The dossier provides:

- Assigned priority
- Inferred severity
- Mismatch category
- Severity delta
- Supporting evidence signals
- Grounded explanation
- Model confidence

All evidence is directly traceable to ticket fields, ensuring compliance with the zero-hallucination requirement defined in the project specification.

In [ ]:
def build_dossier(row):

    evidence = []

    if row["rule_evidence"]:

        for item in str(
            row["rule_evidence"]
        ).split(","):

            evidence.append({

                "signal":"keyword",

                "value":item,

                "weight":"high"
            })

    evidence.append({

        "signal":"resolution_time",

        "value":
        float(
            row[
                "Resolution_Time_Hours"
            ]
        ),

        "interpretation":
        f"Mapped to severity {row['resolution_score']}"
    })

    evidence.append({

        "signal":"llm",

        "value":
        int(
            row["llm_score"]
        )
    })

    evidence.append({

        "signal":"cluster",

        "value":
        int(
            row["cluster_score"]
        )
    })

    return evidence

In [ ]:
import pandas as pd
import json

# ==========================================
# SEVERITY LABEL MAP
# ==========================================

severity_map = {
    0: "Low",
    1: "Medium",
    2: "High",
    3: "Critical"
}

# ==========================================
# LOAD TEST DATA
# ==========================================

df = pd.read_csv(
    "test.csv"
)

dossiers = []

# ==========================================
# GENERATE DOSSIERS
# ==========================================

for _, row in df.iterrows():

    pred, conf = predict_mismatch(
        row["text"]
    )

    dossier = {

        "ticket_id":
        row["Ticket_ID"],

        "assigned_priority":
        row["Priority_Level"],

        "inferred_severity":
        row["inferred_severity"],

        "classifier_prediction":
        int(pred),

        "mismatch_type":
        row["mismatch_type"],

        "severity_delta":
        int(
            row["severity_delta"]
        ),

        "feature_evidence":
        build_dossier(row),

        "constraint_analysis":
        (
            f"The ticket was assigned "
            f"{row['Priority_Level']} priority but was "
            f"inferred as {row['inferred_severity']} severity. "
            f"Supporting signals included "
            f"LLM severity={severity_map[int(row['llm_score'])]}, "
            f"cluster severity={severity_map[int(row['cluster_score'])]}, "
            f"and resolution-time severity={severity_map[int(row['resolution_score'])]}. "
            f"The resulting severity delta was "
            f"{int(row['severity_delta'])}, leading to a "
            f"mismatch classification of "
            f"{row['mismatch_type']}."
        ),

        "confidence":
        round(
            float(conf),
            3
        )
    }

    dossiers.append(
        dossier
    )

# ==========================================
# SAVE JSON
# ==========================================

with open(
    "dossiers.json",
    "w"
) as f:

    json.dump(
        dossiers,
        f,
        indent=2
    )

print(
    f"Generated {len(dossiers)} dossiers"
)

Generated 2000 dossiers


In [ ]:
import json

with open(
    "dossiers.json",
    "r"
) as f:

    dossiers = json.load(f)

allowed_signals = {

    "keyword",
    "resolution_time",
    "llm",
    "cluster"
}

violations = []

for d in dossiers:

    for ev in d["feature_evidence"]:

        if (
            ev["signal"]
            not in
            allowed_signals
        ):

            violations.append(
                (
                    d["ticket_id"],
                    ev["signal"]
                )
            )

print(
    "Signal Violations:",
    len(violations)
)

print(
    violations[:10]
)

Signal Violations: 0
[]


In [ ]:
import json

with open(
    "dossiers.json",
    "r"
) as f:

    dossiers = json.load(f)

scores = []

for d in dossiers:

    score = 0

    if d["feature_evidence"]:
        score += 1

    if d["mismatch_type"]:
        score += 1

    if d["inferred_severity"]:
        score += 1

    if d["constraint_analysis"]:
        score += 1

    if d["confidence"] is not None:
        score += 1

    scores.append(score)

print(
    "Average Completeness:",
    sum(scores)/len(scores),
    "/ 5"
)

Average Completeness: 5.0 / 5


In [ ]:
import pandas as pd
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# =====================================
# LOAD DATA
# =====================================

test_df = pd.read_csv("test.csv")

# =====================================
# PREDICT
# =====================================

preds = []

model.eval()

for text in test_df["text"]:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        pred = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    preds.append(pred)

# =====================================
# TRUE LABELS
# =====================================

labels = test_df["mismatch"].values

# =====================================
# METRICS
# =====================================

acc = accuracy_score(
    labels,
    preds
)

macro_f1 = f1_score(
    labels,
    preds,
    average="macro"
)

precision = precision_score(
    labels,
    preds,
    average="macro"
)

recall = recall_score(
    labels,
    preds,
    average="macro"
)

recall_class0 = recall_score(
    labels,
    preds,
    pos_label=0
)

recall_class1 = recall_score(
    labels,
    preds,
    pos_label=1
)

print("\n===================")
print("TEST RESULTS")
print("===================")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Recall Class0: {recall_class0:.4f}")
print(f"Recall Class1: {recall_class1:.4f}")

print("\nConfusion Matrix")
print(
    confusion_matrix(
        labels,
        preds
    )
)

print("\nClassification Report")
print(
    classification_report(
        labels,
        preds,
        digits=4
    )
)


TEST RESULTS
Accuracy: 0.9175
Macro F1: 0.9161
Precision: 0.9153
Recall: 0.9170
Recall Class0: 0.9131
Recall Class1: 0.9208

Confusion Matrix
[[ 788   75]
 [  90 1047]]

Classification Report
              precision    recall  f1-score   support

           0     0.8975    0.9131    0.9052       863
           1     0.9332    0.9208    0.9270      1137

    accuracy                         0.9175      2000
   macro avg     0.9153    0.9170    0.9161      2000
weighted avg     0.9178    0.9175    0.9176      2000



In [ ]:
test_df["prediction"] = preds

errors = test_df[
    test_df["prediction"] != test_df["mismatch"]
]

print("Total Errors:", len(errors))

errors[
    [
        "Ticket_ID",
        "text",
        "mismatch",
        "prediction",
        "Priority_Level",
        "inferred_severity",
        "mismatch_type"
    ]
].head(20)

Total Errors: 165


,Ticket_ID,text,mismatch,prediction,Priority_Level,inferred_severity,mismatch_type
2,TKT-101353,\nTicket Category: Billing\nTicket Channel: We...,1,0,Low,Medium,Hidden Crisis
29,TKT-102982,\nTicket Category: Fraud\nTicket Channel: Chat...,1,0,High,Critical,Hidden Crisis
41,TKT-115195,\nTicket Category: Billing\nTicket Channel: Em...,1,0,Low,Medium,Hidden Crisis
79,TKT-104103,\nTicket Category: Billing\nTicket Channel: Em...,0,1,Low,Low,Consistent
91,TKT-115629,\nTicket Category: Account\nTicket Channel: Ch...,1,0,Low,Medium,Hidden Crisis
108,TKT-113541,\nTicket Category: Billing\nTicket Channel: Em...,0,1,Low,Low,Consistent
116,TKT-117377,\nTicket Category: Billing\nTicket Channel: Em...,1,0,Medium,Low,False Alarm
118,TKT-104988,\nTicket Category: Billing\nTicket Channel: Ch...,1,0,Medium,Low,False Alarm
120,TKT-118371,\nTicket Category: Account\nTicket Channel: Ch...,1,0,Low,Medium,Hidden Crisis
128,TKT-108735,\nTicket Category: Technical\nTicket Channel: ...,1,0,High,Medium,False Alarm


In [ ]:
test_df["predicted_mismatch"] = preds

test_df.to_csv(
    "test_predictions.csv",
    index=False
)

print(
    "Saved test_predictions.csv"
)

Saved test_predictions.csv


# Deployment and Interactive Analysis

The final system is hosted locally as a Streamlit application.

Users may:

- Analyze individual tickets
- Upload CSV batches
- Generate evidence dossiers
- View mismatch distributions
- Explore severity delta visualizations
- Inspect contributing signals

